# Section 1 - Intro to NLP and LLMs

By the end of this workshop you will have built a **RAG system**: a search engine over every NUS module, connected to an LLM that answers questions using what the search finds. This section lays the groundwork in three parts:

1. **Setup:** load the module data and connect your own Gemini API key.
2. **From text to numbers:** why text is hard for computers, what tokens are, and how an LLM actually writes.
3. **Where LLMs break:** ask Gemini about NUS modules, check its answers against the real data, and see why RAG is the fix.


## Setup

Run the two cells in this part now, while everyone is arriving.

On Colab, the first cell starts by downloading this week's folder from GitHub. Then it loads three things through `rag_utils.py`, a small helper file that all three sections share:

- `modules`: every module NUS offers in AY 2026-2027, taken from [NUSMods](https://nusmods.com). Each module is a Python dictionary with its code, title, description, workload and more. The organisers downloaded this in advance and saved it in `data/`, so the workshop does not depend on NUSMods being up.
- `embeddings`: one row of numbers per module. You will meet these in Section 2.
- `model`: `all-MiniLM-L6-v2`, a small model that turns text into numbers. It runs on your own CPU and needs no API key. The first run downloads it (about 90 MB).

In [ ]:
# %pip install -q google-genai sentence-transformers numpy matplotlib
import json
import os
import re
import sys
import textwrap
import urllib.request
import warnings

# On Colab: download only the week-11-nlp folder of the workshop repo (a few seconds)
if "google.colab" in sys.modules:
    if not os.path.exists("/content/Sem-1"):
        !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/NUS-SDS-Workshops-AY2627/Sem-1.git /content/Sem-1
        !git -C /content/Sem-1 sparse-checkout set week-11-nlp
    %cd /content/Sem-1/week-11-nlp

sys.path.append("..")  # rag_utils.py is one folder up from notebooks/
os.environ["TRANSFORMERS_VERBOSITY"] = os.environ["HF_HUB_VERBOSITY"] = "error"  # hide harmless warnings
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

import rag_utils
from rag_utils import build_chunk, keyword_search, search

modules, embeddings, model = rag_utils.load()
by_code = {module["moduleCode"]: module for module in modules}  # "CS4248" -> its record

print(f"{len(modules)} modules offered in AY {rag_utils.AY}, fetched from NUSMods on {rag_utils.FETCH_DATE}")

### Connect Gemini

Each of you needs your **own** free API key: the free quota is counted per Google Cloud project, so 40 people sharing one key would run out within minutes.

> **Your Gemini API key (one-time, about 2 minutes)**
> 1. Go to **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)**, sign in with a *personal* Google account, find the 🔑 icon, click **Create API key** and copy the key. It's free — **don't** set up billing, because that turns the free tier off for the project.
> 2. In Colab, click the **key icon 🔑 (Secrets)** in the left sidebar → **Add new secret**.
> 3. Name: `GEMINI_API_KEY` · Value: paste your key · switch **Notebook access** on.
>
> No secret set up? The next cell asks you to paste the key instead. Never type your key into the code itself: it would be saved inside the notebook, and anyone you share the notebook with could use up your quota.

The cell below connects `ask_gemini` (from `rag_utils.py`), the one function we use to talk to Gemini. It sends your prompt and nothing else: no extra documents, and no Google Search. If the call fails, for example because the free quota has run out, it prints the reason and returns a saved answer instead of crashing.

In [ ]:
from rag_utils import ask_gemini

# Looks for your key in Colab Secrets first, then asks you to paste it.
rag_utils.connect_gemini()

print("Gemini says:", ask_gemini("Reply with exactly: ready"))

You should see `Gemini says: ready`. If you see `Gemini call failed` instead, the message tells you why. The most common one is `API key not valid`: copy the key again, update the secret, and re-run the cell. If it still fails, raise your hand. You can follow the Gemini demos on the main screen in the meantime.

## 1.1 Reading text

How does a computer read text? Computers only understand numbers, so we have to turn words into numbers. That is hard, because text is:

- **Messy:** punctuation, capital letters, hyphens, and codes like `CS3244`.
- **Ambiguous:** the same word can mean different things, and different words can mean the same thing.

NLP deals with this through a pipeline. We walk through it in this section and finish it in Section 2:


Our example all the way through is one real NUSMods description, the first sentence of **CS3244 Machine Learning**:

> This course introduces basic concepts and algorithms in machine learning and neural networks.

In [ ]:
module = by_code["CS3244"]
sentence = module["description"].split(". ")[0] + "."

print(module["moduleCode"], "-", module["title"])
print(sentence)
print()

# Turning each character into a number is easy...
print([ord(character) for character in "CS3244"])

# ...but the meaning is not in the characters
print("course" == "module")      # same meaning: the description says course, NUSMods says module
print("networks" == "networks")  # same characters: neural networks (CS3244) or computer networks (CS2105)?

## 1.2 Preprocessing

Before a computer can count or compare words, we clean the text up. Each step changes the CS3244 sentence a little:

| Step | What it does |
|---|---|
| Cleaning | remove punctuation and symbols |
| Tokenization | split into words |
| Lowercasing | make every letter lowercase |
| Stopword removal | drop common words such as "this", "and", "in" |
| Stemming | chop off word endings |
| Lemmatization | turn words back into their dictionary form |

The last four steps together are called **normalization**: they make different forms of the same word look alike.

In [ ]:
STOPWORDS = {"a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "in", "is", "of", "on", "or", "some", "the", "this", "to", "with"}

# Real lemmatizers look words up in a full English dictionary. Ours only knows a few words.
LEMMAS = {"introduces": "introduce", "concepts": "concept", "algorithms": "algorithm", "networks": "network",
          "computers": "computer", "systems": "system", "tasks": "task", "students": "student"}


def preprocess(text):
    """Run each preprocessing step on the text and print the result."""
    cleaned = re.sub(r"[^\w\s]", "", text)                         # remove punctuation and symbols
    words = cleaned.split()                                        # split into words
    lowered = [word.lower() for word in words]                     # make every letter lowercase
    kept = [word for word in lowered if word not in STOPWORDS]     # drop common words
    stems = [re.sub(r"(ing|es|s|e)$", "", word) for word in kept]  # chop off a common ending
    lemmas = [LEMMAS.get(word, word) for word in kept]             # look up the dictionary form

    for step, result in [("Raw text", text), ("Cleaning", cleaned), ("Tokenization", words),
                         ("Lowercasing", lowered), ("Stopword removal", kept),
                         ("Stemming", stems), ("Lemmatization", lemmas)]:
        print(f"{step:<17} {result}")


preprocess(sentence)

Stemming is fast but crude: `cours` and `machin` are not words. Lemmatization gives real words, but only for words its dictionary knows.

**YOUR TURN:** run the same steps on another sentence. The default is the third sentence of the CS3244 description: look at what cleaning does to `self-learning`, and what stemming does to `programming`. Then try a sentence from a module you have taken, for example `by_code["CS2040S"]["description"]`.

In [ ]:
# YOUR TURN: change the sentence, then re-run
my_sentence = "The ultimate objective is to build self-learning systems to relieve human from some of already-too-many programming tasks."

preprocess(my_sentence)

Cleaning glued `self-learning` into one word, `selflearning`. Every step throws a little information away, so each one is a trade-off.

These steps prepare text for counting words, which you will meet as TF-IDF in Section 2. Modern language models skip most of them and read text as **tokens** instead.

## 1.3 Tokens

A **token** is a fundamental chunk of text that a language model processes. Splitting on spaces, as in 1.2, is the simplest way to get tokens, but language models use smarter rules:

- Common words are usually 1 token.
- Rare words and codes get split into smaller pieces the model has seen before. On the slides, `CS3244` becomes `CS | 324 | 4`.

Every model comes with its own **tokenizer** that does this splitting, and each one splits a little differently. Here is the tokenizer of `all-MiniLM-L6-v2` reading the full CS3244 description.

In [ ]:
module = by_code["CS3244"]
text = module["description"]

print(module["moduleCode"], "-", module["title"])
print(textwrap.fill(text, 100))
print()

tokens = model.tokenizer.tokenize(text)
print(f"{len(text.split())} words -> {len(tokens)} tokens")
print(tokens)

**YOUR TURN:** alter the words in the list and re-run to see what MiniLM's tokenizer gives. The list starts with words and codes from CS3244, and ends with two oddities from other NUSMods descriptions. Try other module codes, your name, Singlish, or another language.

In [ ]:
# YOUR TURN: alter the words, then re-run
my_words = [
    # from the CS3244 description
    "machine", "regularities", "paradigms", "self-learning",
    # CS3244 and two of its prerequisites
    "CS3244", "CS2040S", "ST2334",
    # oddities from the FIN3702A and CL3211 descriptions
    "eﬃciency", "汉语拼音方案",
]

for word in my_words:
    print(f"{word:>15} -> {model.tokenizer.tokenize(word)}")

# Each token is really an ID number in the model's vocabulary
print()
print("Token IDs:", model.tokenizer.encode("machine learning and neural networks", add_special_tokens=False))

MiniLM lowercases everything first. `##` marks a piece that continues the word before it, so `regularities` becomes `regular` + `##ities`. `[UNK]` means **unknown**: the character is not in the model's vocabulary at all. `eﬃciency` looks normal, but its "ﬃ" is one special character (probably copied from a PDF), so MiniLM cannot read the word. That is messy text in action.

The IDs on the last line are what the model actually receives.

### The token budget

A model can only read a limited number of tokens at once: its **token budget**. Anything past the budget gets cut off, and the model never sees it. Change `budget` below and re-run to see where the CS3244 description gets cut.

In [ ]:
# Try it: change the budget, then re-run
budget = 10

print(f"Read ({budget} tokens):", model.tokenizer.convert_tokens_to_string(tokens[:budget]))
print()
print(f"Cut off ({len(tokens[budget:])} tokens):", model.tokenizer.convert_tokens_to_string(tokens[budget:]))

With the default budget of 10, the model would never learn that CS3244 covers neural networks. Real budgets are bigger, but so is real text. Here is MiniLM's budget, checked against every NUSMods description:

In [ ]:
limit = model.max_seq_length
description_lengths = [len(model.tokenizer.tokenize(m["description"])) for m in modules]
longest = int(np.argmax(description_lengths))

print(f"MiniLM reads at most {limit} tokens")
print(f"CS3244 description: {len(tokens)} tokens")
print(f"Longest description: {description_lengths[longest]} tokens ({modules[longest]['moduleCode']})")
print(f"Descriptions over the limit: {sum(n > limit for n in description_lengths)} of {len(modules)}")

# Gemini uses a different tokenizer, so the same text gives a different count
try:
    gemini_count = rag_utils.gemini_client().models.count_tokens(model=rag_utils.GEMINI_MODEL, contents=text).total_tokens
    print(f"\nThe CS3244 description: MiniLM = {len(tokens)} tokens, Gemini = {gemini_count} tokens")
except Exception:
    pass

## 1.4 How an LLM works

How does an **LLM** (large language model) deal with words? One token at a time:

1. It predicts the next token based on **all the tokens before it**.
2. It picks a likely token, adds it to the text, and repeats.

It is trained to sound **likely**, not to be **true**. That makes it fluent, but also easy to hallucinate.

To watch it happen, we load a tiny LLM called **SmolLM2**. It has 135 million parameters, small enough to run on a laptop CPU (about a 270 MB download). Gemini works the same way, just thousands of times bigger.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

LM_NAME = "HuggingFaceTB/SmolLM2-135M"
lm_tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
lm = AutoModelForCausalLM.from_pretrained(LM_NAME)
lm.eval()


def show_next_tokens(prompt, k=5):
    """Show the model's top k guesses for the very next token."""
    ids = lm_tokenizer(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        probabilities = torch.softmax(lm(ids).logits[0, -1], dim=-1)
    top = torch.topk(probabilities, k)
    print(f"{prompt!r} ...")
    for probability, token_id in zip(top.values, top.indices):
        token = lm_tokenizer.decode([int(token_id)])
        print(f"   {token!r:<16} {probability.item():6.1%}  " + "#" * int(probability.item() * 40))


def generate_step_by_step(prompt, n=15):
    """Generate n tokens one at a time, always picking the most likely one."""
    ids = lm_tokenizer(prompt, return_tensors="pt").input_ids
    for step in range(n):
        with torch.no_grad():
            next_id = int(lm(ids).logits[0, -1].argmax())
        ids = torch.cat([ids, torch.tensor([[next_id]])], dim=1)
        print(f"step {step + 1:>2}: + {lm_tokenizer.decode([next_id])!r}")
    print()
    print("Full text:", lm_tokenizer.decode(ids[0], skip_special_tokens=True))


show_next_tokens("This course introduces basic concepts and algorithms in")

These are SmolLM2's top 5 guesses for the token that comes after the start of the CS3244 sentence. The real next word is `machine`. Now give the model more of the sentence, and watch the guesses change.

In [ ]:
show_next_tokens("This course introduces basic concepts and algorithms in machine")
show_next_tokens("This course introduces basic concepts and algorithms in machine learning and neural")

The more tokens come before, the surer the model gets. Now let it keep going from the shorter prompt. At every step it takes the most likely token and adds it to the text.

In [ ]:
generate_step_by_step("This course introduces basic concepts and algorithms in", n=20)

It reads smoothly, and it sounds like a real course description. But it is not CS3244: the real description continues with *machine learning and neural networks*. The model aims for text that sounds likely, and that is not the same as text that is true.

What happens when we ask about something the model cannot possibly know?

In [ ]:
show_next_tokens("The NUS module CS3244 is titled")
generate_step_by_step("The NUS module CS3244 is titled", n=15)

print()
print("Real title:", by_code["CS3244"]["title"])

## 1.5 Where LLMs break

When we use an LLM to look something up, it might:

- **not know the updated data**, because of its **knowledge cutoff**: it only knows what it was trained on, up to a certain date.
- **invent fluent but untrue facts**, because of **next-token prediction**: it picks what sounds likely, not what is true.

Confident answers that are wrong are called **hallucinations**. SmolLM2 is tiny. Does a real model do better? We will ask Gemini about NUS modules, giving it **no extra information**, and then **check its answer** against the real NUSMods data.

`verify` does the checking. It finds every module code in an answer, such as `CS3244`, and looks it up in this year's module list.

In [ ]:
MODULE_CODE = re.compile(r"\b[A-Z]{2,4}\d{4}[A-Z]{0,2}\b")  # e.g. CS3244, CS2040S


def verify(answer):
    """Look up every module code in the answer in the real NUSMods data."""
    codes = list(dict.fromkeys(MODULE_CODE.findall(answer)))  # unique, in order
    for code in codes:
        real = by_code.get(code)
        if real:
            print(f"  ✅ {code:<9} real title: {real['title']}")
        else:
            print(f"  ❌ {code:<9} not offered in AY {rag_utils.AY}, or does not exist")
        said = next(line for line in answer.splitlines() if code in line)  # where Gemini mentioned it
        print(f"{'':15}Gemini wrote: {textwrap.shorten(re.sub(r'[*_#`|>]', '', said), 90)}")
    print()
    print(f"{sum(code in by_code for code in codes)} of {len(codes)} module codes are real.")

### Ask, then verify

First, the question. Gemini sees only this sentence.

In [ ]:
question = "Which NUS modules cover attention mechanisms? List the module codes and titles."

answer = ask_gemini(question)
display(Markdown(answer))

We verify the answer in the following block. 

In [ ]:
verify(answer)

**How to read it:**

- ❌ means the code is not in this year's NUSMods data. Either the module does not exist, or it is not running this year.
- ✅ with a different title means the code is real, but Gemini attached the wrong module to it, or used an old name.
- ✅ with a matching title means Gemini got that one right. You only know because you checked.

### Numbers it could easily invent

Specific numbers are the easiest thing for an LLM to get plausibly wrong. NUSMods lists the weekly workload of every module in hours.

In [ ]:
module_code = "CS3244"
workload_question = (f"What is the weekly workload of NUS module {module_code} in hours, "
                     "as lecture / tutorial / lab / project / preparation?")

print("GEMINI SAYS:")
print(ask_gemini(workload_question))
print()
print("NUSMODS SAYS:", rag_utils.format_workload(by_code.get(module_code, {}).get("workload")) or "(not listed)")

Hours per week that look reasonable are exactly what next-token prediction produces, whether or not they are true.

Now paste the CS3244 entry from NUSMods into the prompt and ask the same question again.

In [ ]:
entry = build_chunk(by_code.get(module_code, {}))  # code, title, description and workload from NUSMods

print("GEMINI SAYS, WITH THE NUSMODS ENTRY:")
print(ask_gemini(f"Answer using only this entry from NUSMods.\n\n{entry}\n\n{workload_question}"))

With the entry in front of it, Gemini only has to read the answer off, and you know where the answer came from: NUSMods. Hold on to that idea, because it is the core of 1.6. Try a module you have taken, such as `CS2040S` or `ST2334` (two of CS3244's prerequisites), and re-run both cells.

### Right last year, wrong this year

**Context:** a module's *preclusions* are modules that overlap with it so much that you cannot take both. NUSMods updates the list every year, and the AY 2026-2027 list for CS3244 is longer than the AY 2025-2026 one.

In [ ]:
import requests

module_code = "CS3244"
print("GEMINI SAYS:", ask_gemini(f"Which modules are precluded for the NUS module {module_code}? Reply with the module codes only."))

for ay in ["2025-2026", rag_utils.AY]:
    response = requests.get(f"https://api.nusmods.com/v2/{ay}/modules/{module_code}.json", timeout=30)
    preclusion = response.json().get("preclusion") or ""
    print(f"NUSMODS AY {ay}:", ", ".join(dict.fromkeys(MODULE_CODE.findall(preclusion))))

Compare the two NUSMods lines: this year's list has codes that last year's did not. Unless Gemini's training data happened to include this year's list, it cannot know them. That is the knowledge cutoff.

**YOUR TURN:** ask Gemini your own question about NUS modules, then check it. Ask for module codes, so that `verify` has something to check. Some things to try:

- A topic from the CS3244 description: *"Which NUS modules teach neural networks?"*
- What comes next: *"Which NUS modules should I take after CS3244?"*
- A module you have taken: *"What is CS2040S about, and what modules come after it?"*

In [ ]:
# YOUR TURN: ask your own question, then re-run
my_question = "Which NUS modules should I take after CS3244 Machine Learning? List the module codes and titles."

my_answer = ask_gemini(my_question)
display(Markdown(my_answer))
print("-" * 60)
verify(my_answer)

## 1.6 The RAG idea

**Retrieval-Augmented Generation (RAG)** is a framework that improves an LLM by connecting it to an external knowledge source. Ours is NUSMods:

```
 ┌──────┐  (1) ask      ┌─────────┐   (2) query     ┌──────────────────┐
 │      │ ────────────> │         │ ──────────────> │                  │
 │ User │               │ Chatbot │                 │ Knowledge source │
 │      │ <──────────── │  (LLM)  │ <────────────── │    (NUSMods)     │
 └──────┘ (4) response  └─────────┘  (3) knowledge  └──────────────────┘
```

### Why not paste all of NUSMods into the prompt?

Because of the token budget. `build_chunk` turns one module into the text we would paste, like the CS3244 entry in 1.5. The cell below counts tokens with MiniLM's tokenizer, so treat the numbers as estimates.

In [ ]:
chunks = [build_chunk(m) for m in modules]
chunk_lengths = [len(ids) for ids in model.tokenizer(chunks, verbose=False)["input_ids"]]

print(f"CS3244 alone: about {chunk_lengths[modules.index(by_code['CS3244'])]} tokens")
print(f"All {len(modules)} modules: about {sum(chunk_lengths):,} tokens, in every single prompt")
print(f"5 relevant modules: about {sum(chunk_lengths[:5]):,} tokens")

try:
    gemini_limit = rag_utils.gemini_client().models.get(model=rag_utils.GEMINI_MODEL).input_token_limit
    print(f"{rag_utils.GEMINI_MODEL} reads at most {gemini_limit:,} tokens per prompt")
except Exception:
    pass

All of NUSMods would fill most of the budget, if it fits at all, and every single question would pay for it again, when an answer needs only a handful of modules. **So fetch only the few modules that matter, and let the LLM answer from those.** That is **RAG**, retrieval-augmented generation:

```
 ┌─────────────┐     ┌─────────────┐     ┌────────────────────┐     ┌───────────────┐
 │ 1. Question │ ──> │ 2. Retrieve │ ──> │ 3. Augment         │ ──> │ 4. Generate   │
 │  from user  │     │  top-k most │     │  question + those  │     │  LLM answers  │
 │             │     │  relevant   │     │  modules -> prompt │     │  FROM them,   │
 │             │     │  modules    │     │                    │     │  with sources │
 └─────────────┘     └─────────────┘     └────────────────────┘     └───────────────┘
                       Section 2           Section 3                  Section 3
```

### The search process

RAG has to find the right modules first (step 2). The simplest way is **keyword search**: return every module whose title contains your exact words, like the search bar on NUSMods. Search for "machine learning":

In [ ]:
def title_search(query):
    """Modules whose title contains the query, word for word."""
    return [module for module in modules if query.lower() in module["title"].lower()]


found = {module["moduleCode"] for module in title_search("machine learning")}
print(f'"machine learning" in the title: {len(found)} modules')
for code in ["CS3244", "AIS5103"]:
    print(f"    {'found  ' if code in found else 'MISSING'}  {code:<9} {by_code[code]['title']}")

print()
print("AIS5103 description:")
print(textwrap.fill(by_code["AIS5103"]["description"], 100))

Exact, but it ignores modules we might want. AIS5103 is about the same topic, but its title uses different words, so there is no match.

Searching the descriptions instead, like Ctrl+F, catches more. `keyword_search` (from `rag_utils.py`) returns every module whose description contains your text, character for character.

In [ ]:
def print_matches(query, k=5):
    """Run a keyword search and print the first k matches."""
    matches = keyword_search(query)
    print(f'"{query}": {len(matches)} matches')
    for match in matches[:k]:
        print(f"    {match['moduleCode']:<9} {match['title']}")


print_matches("machine learning")
print_matches("learn knowledge (or regularities) from the raw data")  # copied from CS3244
print_matches("learn knowledge from the raw data")                    # the same words, minus "(or regularities)"
print_matches("how computers understand language")                    # a topic in your own words

Ctrl+F on descriptions does find AIS5103, but among 200-plus matches listed alphabetically, not by relevance. And it still compares characters: copy CS3244's words exactly and it finds CS3244; leave out "(or regularities)" and it finds nothing. Describe a topic in your own words and it finds nothing either. To find anything, you have to guess the exact words the author used.

### If you're stuck

Run the cell below. It reloads everything Section 2 needs.

In [ ]:
# Reloads the data and the embedding model if anything above failed
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np

import rag_utils
from rag_utils import ask_gemini, build_chunk, keyword_search, search

modules, embeddings, model = rag_utils.load()
by_code = {module["moduleCode"]: module for module in modules}
print(len(modules), "modules, embeddings", embeddings.shape)